In [ ]:
####### To Be Run Locally

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc

from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModel

from concurrent.futures import ThreadPoolExecutor


In [ ]:
random_state = 23
target_item = 225476 #itemid for Unplanned Cathetar Removal (non patient initiated)


In [ ]:
#Encode, vectorize, and cluster string data for future use
diag_list = pd.read_csv('data/diag_list.csv')
proc_list = pd.read_csv('data/proc_list.csv')

item_list = pd.read_csv('data/item_list.csv')
labitem_list = pd.read_csv('data/labitem_list.csv')


In [ ]:
#BioBERT model for embedding diagnosis, procedure, and item descriptions
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")


In [ ]:
def process_batch(batch):
    batch = [str(item) for item in batch if item is not None]
    
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].numpy()

def generate_embeddings(text_list, batch_size=128):
    batches = [text_list[i:i + batch_size] for i in range(0, len(text_list), batch_size)]
    with ThreadPoolExecutor() as executor:
        embeddings = list(executor.map(process_batch, batches))
    return np.vstack(embeddings)

def process_with_biomedical_embeddings(df, col_to_encode, col_to_embed_and_cluster):
    df[col_to_embed_and_cluster] = df[col_to_embed_and_cluster].fillna('')
    
    label_encoder = LabelEncoder()
    df['Encoding'] = label_encoder.fit_transform(df[col_to_encode])

    embeddings = generate_embeddings(df[col_to_embed_and_cluster].tolist())

    n_uniques = len(df['Encoding'])
    n_clusters = max(1, round(np.sqrt(n_uniques)))

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    df['Cluster'] = kmeans.fit_predict(embeddings)

    df = df[[col_to_encode, 'Encoding', 'Cluster']]

    return df


In [ ]:
processed_diag_list = process_with_biomedical_embeddings(diag_list, 'ICD9_CODE', 'LONG_TITLE')
processed_proc_list = process_with_biomedical_embeddings(proc_list, 'ICD9_CODE', 'LONG_TITLE')
processed_item_list = process_with_biomedical_embeddings(item_list, 'ITEMID', 'LABEL')
processed_labitem_list = process_with_biomedical_embeddings(labitem_list, 'ITEMID', 'LABEL')

processed_diag_list.to_csv('data/processed_diag_list.csv', index=False)
processed_proc_list.to_csv('data/processed_proc_list.csv', index=False)
processed_item_list.to_csv('data/processed_item_list.csv', index=False)
processed_labitem_list.to_csv('data/processed_labitem_list.csv', index=False)


In [ ]:
patients_df = pd.read_csv('data/patients.csv')
initial_subject_df = patients_df[['SUBJECT_ID', 'TRANSFER_FLAG']]
single_column_subject_df = patients_df[['SUBJECT_ID']]


In [ ]:
patient_diag_df = pd.read_csv('data/patient_diags.csv')
patient_proc_df = pd.read_csv('data/patient_procs.csv')


In [ ]:
def flatten_df(patient_df, processed_list, single_column_subject_df=single_column_subject_df, max_cols=25):
    df = patient_df.merge(
        processed_list[['ICD9_CODE', 'Encoding', 'Cluster']],
        on='ICD9_CODE',
        how='left'
    ).drop(columns=['ICD9_CODE'], errors='coerce')

    df.sort_values(by=['SUBJECT_ID', 'SEQ_NUM'], inplace=True)
    df['SEQ_NUM'] = df.groupby('SUBJECT_ID').cumcount() + 1

    num_patients = len(single_column_subject_df)

    encoding_matrix = np.full((num_patients, max_cols), np.nan, dtype=np.int32)
    cluster_matrix = np.full((num_patients, max_cols), np.nan, dtype=np.int32)

    subject_to_idx = {subject: idx for idx, subject in enumerate(single_column_subject_df['SUBJECT_ID'])}

    grouped = df.groupby('SUBJECT_ID')

    for subject_id, group in grouped:
        if subject_id not in subject_to_idx:
            continue

        row_idx = subject_to_idx[subject_id]

        group = group.iloc[:max_cols]

        seq_nums = group['SEQ_NUM'].to_numpy() - 1
        diag_codes = group['Encoding'].to_numpy()
        clusters = group['Cluster'].to_numpy()

        encoding_matrix[row_idx, :len(seq_nums)] = diag_codes[:max_cols]
        cluster_matrix[row_idx, :len(seq_nums)] = clusters[:max_cols]

    encoding_cols = [f'Code_{num:02d}' for num in range(1, max_cols + 1)]
    cluster_cols = [f'Code_{num:02d}_cluster' for num in range(1, max_cols + 1)]

    encoded_clustered_df = pd.concat([
        single_column_subject_df.reset_index(drop=True), 
        pd.DataFrame(encoding_matrix, columns=encoding_cols),
        pd.DataFrame(cluster_matrix, columns=cluster_cols)
        ], axis=1)

    return encoded_clustered_df



In [ ]:
flattened_diag_df = flatten_df(patient_diag_df, processed_diag_list)
flattened_proc_df = flatten_df(patient_proc_df, processed_proc_list)
patient_procevents_df = pd.read_csv('data/patient_procevents.csv')



In [ ]:
ucr_flag_df = patient_procevents_df[patient_procevents_df['ITEMID'] == target_item][['SUBJECT_ID']].drop_duplicates()

ucr_times_df = patient_procevents_df[patient_procevents_df['ITEMID']==target_item]
ucr_times_df = ucr_times_df[['SUBJECT_ID', 'STARTTIME']]
ucr_times_df['STARTTIME'] = pd.to_datetime(ucr_times_df['STARTTIME'], format='%Y-%m-%d %H:%M:%S')
inputevents_cv_df = pd.read_csv('data/inputevents_cv.csv', parse_dates=['CHARTTIME'])
inputevents_mv_df = pd.read_csv('data/inputevents_mv.csv', parse_dates=['STARTTIME'])


In [ ]:
def process_and_flatten_inputevents(
        inputevents_cv_df = inputevents_cv_df,
        inputevents_mv_df = inputevents_mv_df,
        ucr_times_df = ucr_times_df,
        single_column_subject_df = single_column_subject_df,
        processed_item_list = processed_item_list,
        max_items = 200
):

    #Filter and process inputevents, dropping all events AFTER UCR event for applicable patients
    filtered_cv = inputevents_cv_df.merge(ucr_times_df, on='SUBJECT_ID', how='left')
    filtered_cv = filtered_cv[(filtered_cv['STARTTIME'].isna()) | (filtered_cv['CHARTTIME'] <= filtered_cv['STARTTIME'])]
    filtered_cv = filtered_cv.drop(columns=['STARTTIME']).rename(columns={'CHARTTIME': 'STARTTIME'})

    filtered_mv = inputevents_mv_df.merge(ucr_times_df, on='SUBJECT_ID', how='left', suffixes=('_mv', '_ucr'))
    filtered_mv = filtered_mv[(filtered_mv['STARTTIME_ucr'].isna()) | (filtered_mv['STARTTIME_mv'] <= filtered_mv['STARTTIME_ucr'])]
    filtered_mv = filtered_mv.drop(columns=['STARTTIME_ucr']).rename(columns={'STARTTIME_mv': 'STARTTIME'})

    combined_df = pd.concat([filtered_cv, filtered_mv], ignore_index=True)

    #Merge with processed_item_list
    combined_df = combined_df.merge(
        processed_item_list[['ITEMID', 'Encoding', 'Cluster']],
        on='ITEMID',
        how='left'
    )

    #Initialize df for encoding, cluster, amount, and rate
    encoding_matrix = np.full((len(single_column_subject_df), max_items), np.nan, dtype=np.int32)
    cluster_matrix = np.full((len(single_column_subject_df), max_items), np.nan, dtype=np.int32)
    amount_matrix = np.full((len(single_column_subject_df), max_items), np.nan, dtype=np.float32)
    rate_matrix = np.full((len(single_column_subject_df), max_items), np.nan, dtype=np.float32)

    #Map SUBJECT_ID to row index
    subject_to_idx = {subject: idx for idx, subject in enumerate(single_column_subject_df['SUBJECT_ID'])}

    #Group by SUBJECT_ID and sort by STARTTIME
    grouped = combined_df.sort_values(by=['SUBJECT_ID', 'STARTTIME']).groupby('SUBJECT_ID')

    #Populate matrices
    for subject_id, group in grouped:
        if subject_id not in subject_to_idx:
            continue

        row_idx = subject_to_idx[subject_id]
        group = group.iloc[:max_items]  # Limit to max_items

        encoding_matrix[row_idx, :len(group)] = group['Encoding'].to_numpy()
        cluster_matrix[row_idx, :len(group)] = group['Cluster'].to_numpy()
        amount_matrix[row_idx, :len(group)] = group['AMOUNT'].to_numpy()
        rate_matrix[row_idx, :len(group)] = group['RATE'].to_numpy()

    encoding_cols = [f'Code_{num:02d}' for num in range(1, max_items + 1)]
    cluster_cols = [f'Code_{num:02d}_cluster' for num in range(1, max_items + 1)]
    amount_cols = [f'Code_{num:02d}_amount' for num in range(1, max_items + 1)]
    rate_cols = [f'Code_{num:02d}_rate' for num in range(1, max_items + 1)]

    # Step 10: Combine into a single DataFrame
    flattened_df = pd.concat([
        single_column_subject_df.reset_index(drop=True),
        pd.DataFrame(encoding_matrix, columns=encoding_cols),
        pd.DataFrame(cluster_matrix, columns=cluster_cols),
        pd.DataFrame(amount_matrix, columns=amount_cols),
        pd.DataFrame(rate_matrix, columns=rate_cols)
    ], axis=1)

    return flattened_df



In [ ]:
inputevents_df = process_and_flatten_inputevents()


In [ ]:
def process_procevents(
        patient_procevents_df=patient_procevents_df, 
        single_column_subject_df=single_column_subject_df,
        processed_item_list=processed_item_list,
        target_item=target_item,
        max_items=200
        ):
    
    patient_procevents_df['STARTTIME'] = pd.to_datetime(patient_procevents_df['STARTTIME'], format='%Y-%m-%d %H:%M:%S')

    patient_procevents_df = patient_procevents_df.sort_values(by=['SUBJECT_ID', 'STARTTIME'])

    patient_procevents_df = patient_procevents_df.merge(
        processed_item_list[['ITEMID', 'Encoding', 'Cluster']],
        on='ITEMID',
        how='left'
    )

    encoding_matrix = np.full((len(single_column_subject_df), max_items), np.nan, dtype=np.int32)
    cluster_matrix = np.full((len(single_column_subject_df), max_items), np.nan, dtype=np.int32)

    # Map SUBJECT_ID to row index
    subject_to_idx = {subject: idx for idx, subject in enumerate(single_column_subject_df['SUBJECT_ID'])}

    # Group by SUBJECT_ID
    grouped = patient_procevents_df.groupby('SUBJECT_ID')

    for subject_id, group in grouped:
        if subject_id not in subject_to_idx:
            continue

        if target_item in group['ITEMID'].values:
                target_time = group[group['ITEMID'] == target_item]['STARTTIME'].min()
                group = group[group['STARTTIME'] < target_time]

        group = group.iloc[:max_items]

        row_idx = subject_to_idx[subject_id]

        encoding_matrix[row_idx, :len(group)] = group['Encoding'].to_numpy()
        cluster_matrix[row_idx, :len(group)] = group['Cluster'].to_numpy()

    encoding_cols = [f'Code_{num:02d}' for num in range(1, max_items + 1)]
    cluster_cols = [f'Code_{num:02d}_cluster' for num in range(1, max_items + 1)]

    # Combine matrices into a DataFrame
    flattened_df = pd.concat([
        single_column_subject_df.reset_index(drop=True),
        pd.DataFrame(encoding_matrix, columns=encoding_cols),
        pd.DataFrame(cluster_matrix, columns=cluster_cols)
    ], axis=1)

    return flattened_df


In [ ]:
procevents_df = process_procevents()


In [ ]:
outputevents_df = pd.read_csv('data/outputevents.csv')
labevents_df = pd.read_csv('data/labevents.csv')
prescriptions_df = pd.read_csv('data/prescriptions.csv')


In [ ]:
def filter_df_by_starttime(df, time_col, ucr_times_df=ucr_times_df):
    merged_df = df.merge(ucr_times_df, on='SUBJECT_ID', how='left')

    rows_to_drop_count = merged_df[(~merged_df['STARTTIME'].isna()) & (merged_df[time_col] > merged_df['STARTTIME'])].shape[0]
    print(f"Number of rows to be dropped: {rows_to_drop_count}")

    #Keep rows where CHARTTIME <= STARTTIME
    filtered_df = merged_df[(merged_df['STARTTIME'].isna()) | (merged_df[time_col] <= merged_df['STARTTIME'])]

    filtered_df = filtered_df.drop(columns=['STARTTIME']).rename(columns={time_col: 'STARTTIME'})

    return filtered_df


In [ ]:
filtered_outputevents_df = filter_df_by_starttime(outputevents_df, 'CHARTTIME')
filtered_labevents_df = filter_df_by_starttime(labevents_df, 'CHARTTIME')
filtered_prescriptions_df = filter_df_by_starttime(prescriptions_df, 'STARTDATE')


In [ ]:
num_unique_values = labevents_df['ITEMID'].nunique()

num_unique_values

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Group by SUBJECT_ID and count ITEMIDs
itemid_counts = labevents_df.groupby('SUBJECT_ID')['ITEMID'].count()

# Summary statistics
print(itemid_counts.describe())

In [ ]:
# Plot the distribution
plt.figure(figsize=(10, 6))
plt.hist(itemid_counts, bins=30, edgecolor='k', alpha=0.7)
plt.title('Distribution of Number of ITEMIDs per SUBJECT_ID')
plt.xlabel('Number of ITEMIDs')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()



In [ ]:
def process_event_df(event_df, processed_list, identifier='ITEMID', time_col='STARTTIME', val=False, extra_col=False, single_column_subject_df=single_column_subject_df):
    df = event_df.sort_values(by=['SUBJECT_ID', time_col]).drop(columns=[time_col])

    df = df.merge(
        processed_list[[identifier]],
        on=identifier,
        how="left",
    )

    unique_itemids = df[identifier].unique()

    subject_to_idx = {subject: idx for idx, subject in enumerate(single_column_subject_df["SUBJECT_ID"])}
    itemid_to_idx = {itemid: idx for idx, itemid in enumerate(unique_itemids)}
    
    flag_matrix = np.zeros((len(single_column_subject_df), len(unique_itemids)), dtype=np.int32)
    count_matrix = np.zeros((len(single_column_subject_df), len(unique_itemids)), dtype=np.int32)

    if val:
        avg_val_matrix = np.zeros((len(single_column_subject_df), len(unique_itemids)), dtype=np.float32)

    if extra_col:    
        extra_col_matrix = np.zeros((len(single_column_subject_df), len(unique_itemids)), dtype=np.int32)

    for subject_id, group in df.groupby("SUBJECT_ID"):
        if subject_id not in subject_to_idx:
            continue

        row_idx = subject_to_idx[subject_id]

        counts = group[identifier].value_counts().reindex(unique_itemids, fill_value=0).to_numpy()

        flag_matrix[row_idx, :] = 1
        count_matrix[row_idx, :] += counts

        if val:
            avg_vals = group.groupby(identifier)[val].mean().reindex(unique_itemids, fill_value=0).to_numpy()
            avg_val_matrix[row_idx, :] += avg_vals

        if extra_col:
            extra_col_flag = group.groupby(identifier)[extra_col].apply(
                lambda x: 1 if "abnormal" in x.values else 0
            ).reindex(unique_itemids, fill_value=0).to_numpy()
            extra_col_matrix[row_idx, :] += extra_col_flag

    if val:
        avg_val_matrix = np.divide(
            avg_val_matrix,
            count_matrix,
            out=np.zeros_like(avg_val_matrix),
            where=count_matrix != 0,
        )

    flag_cols = [f"{itemid}_flag" for itemid in unique_itemids]
    count_cols = [f"{itemid}_count" for itemid in unique_itemids]

    dataframes_to_concat = [
    single_column_subject_df.reset_index(drop=True),
    pd.DataFrame(flag_matrix, columns=flag_cols),
    pd.DataFrame(count_matrix, columns=count_cols),
    ]

    if val:
        avg_val_cols = [f"{itemid}_avg_val" for itemid in unique_itemids]
        dataframes_to_concat.append(pd.DataFrame(avg_val_matrix, columns=avg_val_cols))

    if extra_col:
        extra_col_cols = [f"{itemid}_abnormal" for itemid in unique_itemids]
        dataframes_to_concat.append(pd.DataFrame(extra_col_matrix, columns=extra_col_cols))

    final_df = pd.concat(dataframes_to_concat, axis=1)

    return final_df


In [ ]:
unique_prescriptions_list = pd.DataFrame(prescriptions_df['DRUG'].unique(), columns=['DRUG'])
processed_prescriptions_list = process_with_biomedical_embeddings(unique_prescriptions_list, 'DRUG', 'DRUG')
processed_labevents_df = process_event_df(filtered_labevents_df, processed_labitem_list, val='VALUENUM', extra_col='FLAG')
processed_outputevents_df = process_event_df(filtered_outputevents_df, processed_item_list, val='VALUE')
processed_prescriptions_df = process_event_df(prescriptions_df, processed_prescriptions_list, identifier='DRUG', time_col='STARTDATE')


In [ ]:
dataframes_with_prefixes = [
    (flattened_diag_df, "diag_"),
    (flattened_proc_df, "proc_"),
    (inputevents_df, "ie_"),
    (procevents_df, "pe_"),
    (processed_labevents_df, "le_"),
    (processed_outputevents_df, "oe_"),
    (processed_prescriptions_df, "rx_"),
]

final_df = initial_subject_df.copy()

# Join each dataframe with the specified prefix
for df, prefix in dataframes_with_prefixes:
    # Rename all columns except 'SUBJECT_ID' with the prefix
    df_prefixed = df.rename(columns={col: f"{prefix}{col}" for col in df.columns if col != "SUBJECT_ID"})
    
    try:
        # Merge with final_df
        final_df = final_df.merge(df_prefixed, on="SUBJECT_ID", how="left")
        print(f"Successfully merged: {prefix}")

        # Delete the intermediate dataframe to free up memory
        del df, df_prefixed
        gc.collect()

    except Exception as e:
        print(f"Failed to merge: {prefix}")
        print(f"Error: {e}")

# Add the UCR_Flag column
ucr_flag_set = set(ucr_flag_df["SUBJECT_ID"])
final_df["UCR_Flag"] = final_df["SUBJECT_ID"].isin(ucr_flag_set).astype(np.int32)


In [ ]:
final_df.to_parquet('data/model_inputs.parquet', index=False, compression='snappy')